In [39]:
import time
import pyautogui as pyg

# =========================================================
# 1. 座標與區域截圖設定
# =========================================================
baseX = 518      # 第一顆寶石 (row 0, col 0) 的 X 座標
baseY = 384      # 第一顆寶石 (row 0, col 0) 的 Y 座標
gap = 40         # 寶石中心間距 (像素)
GAME_TIME = 65   # 遊玩時間 (秒)

pyg.PAUSE = 0.0001  # 微秒級滑鼠輸出

# 8x8 棋盤極速區域截圖範圍
REGION_LEFT = baseX - 15
REGION_TOP = baseY - 15
REGION_WIDTH = 8 * gap + 30
REGION_HEIGHT = 8 * gap + 30

banned_moves = {}

def get_gem_position(r, c):
    """計算螢幕絕對 X, Y 座標"""
    return (baseX + c * gap, baseY + r * gap)

def get_gem_color_offset(r, c, im):
    """
    偏心採樣：取寶石中心偏上方 4 像素
    避開 2x, 3x 倍數寶石數字白字
    """
    rel_x = c * gap + 15
    rel_y = (r * gap + 15) - 4
    try:
        return im.getpixel((rel_x, rel_y))
    except Exception:
        return (0, 0, 0)

# =========================================================
# 2. 精準色彩比對與【魔方 (Hypercube) 特徵辨識】
# =========================================================
def is_background(color):
    """亮度過低判定為背景或下落黑影 (防字串傳入型態錯誤)"""
    if color is None or isinstance(color, str):
        return color is None
    return (color[0] + color[1] + color[2]) < 85

def is_hypercube(color):
    """
    【魔方專屬特徵辨識】
    旋轉魔方的底色為灰棕/彩虹混色：
    1. R, G, B 三色極度接近 (最大色差 max_diff < 15，排除純色寶石)
    2. 總亮度介於 260 ~ 460 之間 (排除過暗黑影與過亮白光)
    """
    if is_background(color):
        return False
    r, g, b = color[:3]
    max_diff = max(abs(r - g), abs(g - b), abs(b - r))
    brightness = r + g + b
    return (max_diff < 15) and (260 < brightness < 460)

def colors_match(c1, c2, tolerance=18):
    """色彩比對 (加入字串防護)"""
    if is_background(c1) or is_background(c2):
        return False
    if isinstance(c1, str) or isinstance(c2, str):
        return False
    return all(abs(a - b) <= tolerance for a, b in zip(c1[:3], c2[:3]))

# =========================================================
# 3. 官方高分權重矩陣 + 魔方最高權重引爆
# =========================================================
def check_match_at(r, c, board):
    color = board[r][c]
    # 魔方與背景不參與普通 3 連線連鎖計算
    if is_background(color) or color == "HYPERCUBE":
        return 0

    h_count = 1
    nc = c - 1
    while nc >= 0 and colors_match(board[r][nc], color):
        h_count += 1
        nc -= 1
    nc = c + 1
    while nc < 8 and colors_match(board[r][nc], color):
        h_count += 1
        nc += 1

    v_count = 1
    nr = r - 1
    while nr >= 0 and colors_match(board[nr][c], color):
        v_count += 1
        nr -= 1
    nr = r + 1
    while nr < 8 and colors_match(board[nr][c], color):
        v_count += 1
        nr += 1

    if h_count >= 5 or v_count >= 5: return 10000
    if h_count >= 3 and v_count >= 3: return 7000
    if h_count >= 4 or v_count >= 4: return 4000
    if h_count >= 3 or v_count >= 3: return 1000
    return 0

def evaluate_move(r, c, dr, dc, board, current_time):
    nr, nc = r + dr, c + dc
    if not (0 <= nr < 8 and 0 <= nc < 8):
        return 0

    # 檢查是否在冷卻黑名單內
    m_key1 = (r, c, nr, nc)
    m_key2 = (nr, nc, r, c)
    if m_key1 in banned_moves and current_time < banned_moves[m_key1]:
        return 0
    if m_key2 in banned_moves and current_time < banned_moves[m_key2]:
        return 0

    c1 = board[r][c]
    c2 = board[nr][nc]

    if is_background(c1) or is_background(c2):
        return 0

    # 【魔方核心邏輯】：
    # 只要兩者之中有一個是魔方，且另一個是實體寶石，賦予全場最高 30,000 分，無視連線立即引爆！
    if (c1 == "HYPERCUBE" and c2 != "HYPERCUBE") or (c2 == "HYPERCUBE" and c1 != "HYPERCUBE"):
        return 30000

    if colors_match(c1, c2):
        return 0

    # 模擬一般寶石盤面交換
    sim_board = [list(row) for row in board]
    sim_board[r][c], sim_board[nr][nc] = sim_board[nr][nc], sim_board[r][c]

    score1 = check_match_at(r, c, sim_board)
    score2 = check_match_at(nr, nc, sim_board)

    base_score = max(score1, score2)
    if base_score == 0:
        return 0

    # 底部塌陷加權
    bottom_weight = (max(r, nr) + 1) * 70
    return base_score + bottom_weight

# =========================================================
# 4. 主程式迴圈 (原版穩定機制 + 魔方爆破)
# =========================================================
def main():
    print(f"基準座標設定: baseX={baseX}, baseY={baseY}")
    print("程式將在 3 秒後開始執行，請切換至 Bejeweled Blitz 遊戲視窗...")
    pyg.sleep(3)
    print("【啟動原版高穩定 + 魔方秒速引爆模式！】")

    start_time = time.time()
    last_action_time = time.time()
    directions = ((0, 1), (1, 0), (0, -1), (-1, 0))

    while time.time() - start_time < GAME_TIME:
        now = time.time()
        
        # 1. 極速區域截圖
        im = pyg.screenshot(region=(REGION_LEFT, REGION_TOP, REGION_WIDTH, REGION_HEIGHT))
        
        # 2. 建立盤面 (偏心採樣 + 魔方辨識)
        board = []
        for r in range(8):
            row = []
            for c in range(8):
                color = get_gem_color_offset(r, c, im)
                if is_hypercube(color):
                    row.append("HYPERCUBE")  # 標記為魔方
                else:
                    row.append(color)
            board.append(row)

        possible_moves = []
        for r in range(8):
            for c in range(8):
                for dr, dc in directions:
                    weight = evaluate_move(r, c, dr, dc, board, now)
                    if weight > 0:
                        possible_moves.append((weight, r, c, dr, dc))

        if possible_moves:
            possible_moves.sort(key=lambda x: x[0], reverse=True)
            w, r, c, dr, dc = possible_moves[0]
            nr, nc = r + dr, c + dc
            
            # 冷卻鎖定 2.5 秒
            banned_moves[(r, c, nr, nc)] = now + 2.5
            
            x1, y1 = get_gem_position(r, c)
            x2, y2 = get_gem_position(nr, nc)
            
            # 點擊交換
            pyg.click(x1, y1)
            pyg.click(x2, y2)
            
            # 保留原版穩定的 0.08 秒手速緩衝
            pyg.sleep(0.08)
            last_action_time = time.time()
        else:
            if now - last_action_time > 0.15:
                banned_moves.clear()
                pyg.sleep(0.03)
                last_action_time = time.time()
            else:
                pyg.sleep(0.01)

    print("【遊戲時間結束！】")

if __name__ == "__main__":
    main()

基準座標設定: baseX=518, baseY=384
程式將在 3 秒後開始執行，請切換至 Bejeweled Blitz 遊戲視窗...
【啟動原版高穩定 + 魔方秒速引爆模式！】
【遊戲時間結束！】
